# Practice - 06 - Structured Streaming restart and event-time reasoning

Prerequisites / execution context: Run 00. Uses a dedicated append-only Delta source and AvailableNow. No permanently running query.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R29: Structured Streaming concepts](https://docs.databricks.com/aws/en/structured-streaming/concepts); [R30: Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints); [R31: Watermarks](https://docs.databricks.com/aws/en/structured-streaming/watermarks); [R04: Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Restart an append stream

Create an isolated Delta source SRC(id INT,event_ts TIMESTAMP), destination DST, checkpoint CP. Insert id=1 at 2026-11-01 10:00. Implement drain() with AvailableNow. Drain twice, append id=2 at 10:01, drain again. Verify progress is retained.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
import uuid
rid = uuid.uuid4().hex[:10]
SRC = f"{BRONZE}.stream_src_{rid}"
DST = f"{SILVER}.stream_dst_{rid}"
CP = f"{VOLUME}/streams/{rid}/checkpoint"

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert spark.table(DST).count() == 2
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIENyZWF0ZSBhIHNvdXJjZSwgc2luaywgYW5kIGNoZWNrcG9pbnQKVGhpcyBkZW1vbnN0cmF0ZXMgaW5jcmVtZW50YWwgYXBwZW5kIHByb2Nlc3NpbmcuIFF1ZXJ5IGhpc3RvcnkgYmVsb25ncyB0byB0aGUgY2hlY2twb2ludCwgbm90IHRvIHRoZSBub3RlYm9vayBQeXRob24gcHJvY2Vzcy4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmltcG9ydCB1dWlkCnJpZD11dWlkLnV1aWQ0KCkuaGV4WzoxMF0KU1JDPWYmcXVvdDt7QlJPTlpFfS5zdHJlYW1fc3JjX3tyaWR9JnF1b3Q7OyBEU1Q9ZiZxdW90O3tTSUxWRVJ9LnN0cmVhbV9kc3Rfe3JpZH0mcXVvdDsKQ1A9ZiZxdW90O3tWT0xVTUV9L3N0cmVhbXMve3JpZH0vY2hlY2twb2ludCZxdW90OwpzcGFyay5zcWwoZiZxdW90O0NSRUFURSBUQUJMRSB7U1JDfSAoaWQgSU5ULGV2ZW50X3RzIFRJTUVTVEFNUCkgVVNJTkcgREVMVEEmcXVvdDspCnNwYXJrLnNxbChmJnF1b3Q7SU5TRVJUIElOVE8ge1NSQ30gVkFMVUVTICgxLFRJTUVTVEFNUCAmI3gyNzsyMDI2LTExLTAxIDEwOjAwOjAwJiN4Mjc7KSZxdW90OykKZGVmIGRyYWluKCk6CiAgICBxPShzcGFyay5yZWFkU3RyZWFtLnRhYmxlKFNSQykud3JpdGVTdHJlYW0ub3B0aW9uKCZxdW90O2NoZWNrcG9pbnRMb2NhdGlvbiZxdW90OyxDUCkKICAgICAgIC50cmlnZ2VyKGF2YWlsYWJsZU5vdz1UcnVlKS50b1RhYmxlKERTVCkpCiAgICBxLmF3YWl0VGVybWluYXRpb24oKQogICAgcmV0dXJuIHEKcT1kcmFpbigpCmFzc2VydCBzcGFyay50YWJsZShEU1QpLmNvdW50KCk9PTEKZHJhaW4oKQphc3NlcnQgc3BhcmsudGFibGUoRFNUKS5jb3VudCgpPT0xCnNwYXJrLnNxbChmJnF1b3Q7SU5TRVJUIElOVE8ge1NSQ30gVkFMVUVTICgyLFRJTUVTVEFNUCAmI3gyNzsyMDI2LTExLTAxIDEwOjAxOjAwJiN4Mjc7KSZxdW90OykKZHJhaW4oKQphc3NlcnQgc3BhcmsudGFibGUoRFNUKS5jb3VudCgpPT0yCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Coding task 2: Plan a stateful query

Read SRC as a stream, use a 10-minute event_ts watermark, aggregate a count in five-minute windows, assign stateful, and explain the plan. Do not start a second writer with CP.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIEJ1aWxkIGEgd2F0ZXJtYXJrIHF1ZXJ5IHBsYW4gd2l0aG91dCBydW5uaW5nIGEgc2Vjb25kIHN0cmVhbQpBIDEwLW1pbnV0ZSB3YXRlcm1hcmsgaXMgcmVsYXRpdmUgdG8gb2JzZXJ2ZWQgZXZlbnQgdGltZSwgbm90IHdhbGwtY2xvY2sgdGltZS4gRXZlbnRzIHdpdGhpbiB0aGUgc3VwcG9ydGVkIGxhdGVuZXNzIHRocmVzaG9sZCBhcmUgcHJvdGVjdGVkOyBvbGRlciBvbmVzIGFyZSBub3QgZ3VhcmFudGVlZCB0byBiZSBwcm9jZXNzZWQuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9zdHJ1Y3R1cmVkLXN0cmVhbWluZy93YXRlcm1hcmtzIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjMxOiBXYXRlcm1hcmtzPC9hPi4KCgpUaGUgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uIGlzIHNob3duIG5leHQuIENvbXBhcmUgeW91ciBkZXNpZ24gYW5kIHJlc3VsdHMgYmVmb3JlIGNvcHlpbmcgaXQuPC9kaXY+PC9kZXRhaWxzPg==')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPnN0YXRlZnVsPShzcGFyay5yZWFkU3RyZWFtLnRhYmxlKFNSQykud2l0aFdhdGVybWFyaygmcXVvdDtldmVudF90cyZxdW90OywmcXVvdDsxMCBtaW51dGVzJnF1b3Q7KQogICAgICAgICAgLmdyb3VwQnkoRi53aW5kb3coJnF1b3Q7ZXZlbnRfdHMmcXVvdDssJnF1b3Q7NSBtaW51dGVzJnF1b3Q7KSkuY291bnQoKSkKc3RhdGVmdWwuZXhwbGFpbigpCiMgQSByZWFsIHdyaXRlciBmb3IgdGhpcyBkaWZmZXJlbnQgcXVlcnkgbXVzdCB1c2UgYSBESUZGRVJFTlQgY2hlY2twb2ludC4KPC9jb2RlPjwvcHJlPjwvZGV0YWlscz4=')


## Written / design task 1

For observed event time 12:20 and a ten-minute lateness threshold, reason about events at 12:15 and 12:01. Can you update the append source without changing the consumption design? Can you change a stateful query while keeping its checkpoint?


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgQ2hhbGxlbmdlcwoxLiBTb3VyY2UgbWF4IGV2ZW50IHRpbWUgaXMgMTI6MjAgYW5kIGxhdGVuZXNzIGlzIDEwIG1pbnV0ZXM6IHJlYXNvbiBhYm91dCAxMjoxNSBhbmQgMTI6MDEgZXZlbnRzLiBUaGUgZmlyc3QgaXMgd2l0aGluIHRoZSB0aHJlc2hvbGQ7IHRoZSBzZWNvbmQgaXMgb3V0c2lkZSB0aGUgZ3VhcmFudGVlZCByYW5nZS4gQWN0dWFsIHdhdGVybWFyayBhZHZhbmNlbWVudCBvY2N1cnMgYWNyb3NzIGJhdGNoZXMuCjIuIFdoeSBub3QgdXNlIHRoaXMgYXBwZW5kIHN0cmVhbSB1bmNoYW5nZWQgYWZ0ZXIgVVBEQVRFIG9uIHRoZSBzb3VyY2U/IEFwcGVuZC1vbmx5IHByb2Nlc3NpbmcgaXMgbm90IGEgZ2VuZXJhbCBDREMgc29sdXRpb24uIFN0dWR5IENERi9BVVRPIENEQyBpbiBsYWIgMTIuCjMuIFdoeSBub3Qgc3dpdGNoIHRoaXMgY2hlY2twb2ludCB0byBhIGRpZmZlcmVudCBhZ2dyZWdhdGlvbj8gU3RhdGVmdWwgcXVlcnkgY2hhbmdlcyBoYXZlIHJlc3RhcnQgY29tcGF0aWJpbGl0eSByZXN0cmljdGlvbnMuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9zdHJ1Y3R1cmVkLXN0cmVhbWluZy9jaGVja3BvaW50cyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIzMDogU3RyZWFtaW5nIGNoZWNrcG9pbnRzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi90YWJsZXMvZmVhdHVyZXMvY2hhbmdlLWRhdGEtZmVlZCIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlI2ODogQ2hhbmdlIGRhdGEgZmVlZDwvYT4uCjwvZGl2PjwvZGV0YWlscz4=')
